In [ ]:
#training the maodel

import os
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# ==========================================================
# CONFIG
# ==========================================================
FEATURE_DIR = "/Users/prajwalnara/Documents/EEG-motor-imagery/data/features/band_power_10sub"
SUBJECTS = [
    "S001", "S002", "S003", "S004", "S005",
    "S006", "S007", "S008", "S009", "S010",
]

# Subject-level split: subjects in TEST_SUBJECTS are held out completely —
# none of their epochs (from either R04/R08 or R12) appear in training.
# This is the real generalization test Stage 2 couldn't give you with
# only one subject. ~30% held out -> 3 of 10 subjects.
TEST_SUBJECTS = {"S008", "S009", "S010"}
TRAIN_SUBJECTS = [s for s in SUBJECTS if s not in TEST_SUBJECTS]

CANONICAL_EVENT_ID = {"rest": 1, "left": 2, "right": 3}


# ==========================================================
# LOAD + POOL ALL SUBJECTS
# ==========================================================
def load_subject_data(subject):
    """
    Loads one subject's saved train+test features (their R04+R08 epochs
    and R12 epochs respectively, from the extraction step) and remaps
    labels to CANONICAL_EVENT_ID using that subject's own saved mapping,
    so label integers are guaranteed consistent across subjects even if
    MNE assigned different codes per subject.
    """
    save_dir = os.path.join(FEATURE_DIR, subject)

    X_train = np.load(os.path.join(save_dir, "X_train.npy"))
    y_train_raw = np.load(os.path.join(save_dir, "y_train.npy"))
    X_test = np.load(os.path.join(save_dir, "X_test.npy"))
    y_test_raw = np.load(os.path.join(save_dir, "y_test.npy"))
    event_id_map = np.load(
        os.path.join(save_dir, "event_id_map.npy"), allow_pickle=True
    ).item()

    # Remap this subject's raw codes -> canonical codes via name lookup,
    # in case events_from_annotations assigned different integers here
    # than for another subject.
    code_to_name = {v: k for k, v in event_id_map.items()}
    remap = {orig_code: CANONICAL_EVENT_ID[name] for orig_code, name in code_to_name.items()}

    y_train = np.array([remap[c] for c in y_train_raw])
    y_test = np.array([remap[c] for c in y_test_raw])

    # This subject contributes ALL of its epochs (both the R04+R08 set
    # and the R12 set) to whichever side of the subject-level split it
    # lands on — we pool both together here since the split happens at
    # the subject level, not the run level, in this stage.
    X_all = np.concatenate([X_train, X_test], axis=0)
    y_all = np.concatenate([y_train, y_test], axis=0)
    subject_ids = np.full(len(y_all), subject)

    return X_all, y_all, subject_ids


X_list, y_list, subj_list = [], [], []
loaded, skipped = [], []

for subject in SUBJECTS:
    try:
        X_s, y_s, subj_s = load_subject_data(subject)
        X_list.append(X_s)
        y_list.append(y_s)
        subj_list.append(subj_s)
        loaded.append(subject)
    except FileNotFoundError as e:
        print(f"Skipping {subject}, feature files not found: {e}")
        skipped.append(subject)

X_all = np.concatenate(X_list, axis=0)
y_all = np.concatenate(y_list, axis=0)
subject_ids_all = np.concatenate(subj_list, axis=0)

print(f"Loaded {len(loaded)} subjects, skipped {len(skipped)}: {skipped}")
print(f"Pooled data: X={X_all.shape}, y={y_all.shape}")

# ==========================================================
# SUBJECT-LEVEL SPLIT (the actual point of this stage)
# ==========================================================
train_mask = np.isin(subject_ids_all, TRAIN_SUBJECTS)
test_mask = np.isin(subject_ids_all, list(TEST_SUBJECTS))

X_train, y_train = X_all[train_mask], y_all[train_mask]
X_test, y_test = X_all[test_mask], y_all[test_mask]

print(f"\nTrain subjects ({len(TRAIN_SUBJECTS)}): {TRAIN_SUBJECTS}")
print(f"Test subjects  ({len(TEST_SUBJECTS)}): {sorted(TEST_SUBJECTS)}")
print(f"Train set: X={X_train.shape}, label counts={dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Test set:  X={X_test.shape}, label counts={dict(zip(*np.unique(y_test, return_counts=True)))}")

# ==========================================================
# TRAIN MODEL
# ==========================================================
model = Pipeline([
    ("scaler", StandardScaler()),
    ("lda", LinearDiscriminantAnalysis()),
])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# ==========================================================
# EVALUATE — OVERALL
# ==========================================================
code_to_name = {v: k for k, v in CANONICAL_EVENT_ID.items()}
labels_sorted = sorted(CANONICAL_EVENT_ID.values())
label_names = [code_to_name[c] for c in labels_sorted]

acc = accuracy_score(y_test, y_pred)
print(f"\n========== OVERALL TEST ACCURACY: {acc:.4f} ==========")
print("\nConfusion Matrix (rows=true, cols=predicted)")
print("Labels order:", label_names)
print(confusion_matrix(y_test, y_pred, labels=labels_sorted))
print("\nClassification Report")
print(classification_report(y_test, y_pred, labels=labels_sorted, target_names=label_names))

# ==========================================================
# EVALUATE — PER TEST SUBJECT (variance matters, not just mean)
# ==========================================================
print("\n========== PER-SUBJECT TEST ACCURACY ==========")
test_subject_ids = subject_ids_all[test_mask]
for subject in sorted(TEST_SUBJECTS):
    s_mask = test_subject_ids == subject
    s_acc = accuracy_score(y_test[s_mask], y_pred[s_mask])
    print(f"  {subject}: {s_acc:.4f}  (n={s_mask.sum()})")

Skipping S001, feature files not found: [Errno 2] No such file or directory: '/Users/prajwalnara/Documents/EEG-motor-imagery/data/features/band_power_10sub/S001/X_train.npy'
Skipping S002, feature files not found: [Errno 2] No such file or directory: '/Users/prajwalnara/Documents/EEG-motor-imagery/data/features/band_power_10sub/S002/X_train.npy'
Skipping S003, feature files not found: [Errno 2] No such file or directory: '/Users/prajwalnara/Documents/EEG-motor-imagery/data/features/band_power_10sub/S003/X_train.npy'
Skipping S004, feature files not found: [Errno 2] No such file or directory: '/Users/prajwalnara/Documents/EEG-motor-imagery/data/features/band_power_10sub/S004/X_train.npy'
Skipping S005, feature files not found: [Errno 2] No such file or directory: '/Users/prajwalnara/Documents/EEG-motor-imagery/data/features/band_power_10sub/S005/X_train.npy'
Skipping S006, feature files not found: [Errno 2] No such file or directory: '/Users/prajwalnara/Documents/EEG-motor-imagery/data/

ValueError: need at least one array to concatenate